# Tratado 1 (corregido) — DENTEX → YOLO (2 clases): diagnóstico del error y pipeline reparado

Artículo II: YOLO11n vs. YOLO26n en radiografías panorámicas. Clases: `periapical_lesion` (0), `impacted_tooth` (1).

---

## 1. Diagnóstico: por qué las cajas salían mal

**La causa casi seguro NO es la fórmula COCO → YOLO ni las etiquetas. Es el emparejamiento imagen ↔ JSON.**

DENTEX trae 4 carpetas y **cada una tiene su propia numeración `train_0.png … train_N.png`**:

| Carpeta | Imágenes | Nota |
|---|---|---|
| `quadrant/xrays` | 693 | `train_0.png` es una radiografía |
| `quadrant_enumeration/xrays` | 634 | `train_0.png` es **otra** radiografía |
| `quadrant-enumeration-disease/xrays` | 705 | `train_0.png` es **otra** (la que necesitamos) |
| `unlabelled/xrays` | 1571 | `train_0.png` es **otra** |

El código anterior hacía esto (celda 4):

In [ ]:
for p in RAW_ROOT.rglob("*"):
    disk_images.setdefault(p.name, p)   # ← el primer "train_0.png" que aparezca gana

Así, cuando el JSON de *disease* decía `train_0.png`, se le asignaba el `train_0.png` de **cualquier otra carpeta**. Por eso la caja no cae sobre ningún diente.

**Evidencia en tus propios logs** (no hacía falta nada más):

- `Imágenes con tamaño JSON != tamaño real: 657` de 705 (93 %). Si el emparejamiento fuera correcto serían ~0.
- `JSON sin archivo: 0` era un falso "todo bien": como los nombres existen en las 4 carpetas, nunca falla el lookup.
- `Imágenes en disco: 1571` = los nombres únicos de `unlabelled`, que contiene todos los índices.

**Corrección:** emparejar siempre por **carpeta del JSON**: `IMG_DIR = carpeta_del_json / "xrays"`, y además extraer solo esa carpeta (el zip pesa 10.9 GB; la mayoría no sirve).

**Mejoras adicionales incluidas:**

1. Validación automática del formato del `bbox` contra el polígono `segmentation` (no se asume xywh, se comprueba).
2. `assert` de que <1 % de imágenes tienen tamaño distinto al del JSON (si vuelve a fallar, el notebook se detiene).
3. Verificación *round-trip*: se reconstruyen las cajas desde los `.txt` YOLO y se comparan con el JSON original (error < 1 px).
4. Verificación visual lado a lado: JSON crudo vs. label YOLO escrito en disco.

---

## 2. Decisiones que debes confirmar

| Decisión | Valor por defecto | Comentario |
|---|---|---|
| `INCLUDE_NEGATIVES` | `True` | Hay 379 radiografías con solo caries/caries profunda: son negativas **válidas** para nuestras 2 clases (el dataset anota las 4 categorías en cada imagen). Sirven para medir falsos positivos y reflejan el uso real del prototipo (el usuario sube cualquier panorámica). Si prefieres solo positivas, ponlo en `False` (326 imágenes). |
| Estratificación | por combinación de clases (`perio`, `imp`, `ambas`, `ninguna`) | Semilla 42, split único. |
| Fuga por paciente | No controlable | DENTEX no publica ID de paciente; se declara como limitación en el artículo. |
| Lesión periapical | solo 158 instancias | En test habrá ~25–35: los IC 95 % serán anchos. Reportarlo con honestidad. |
| Resolución de entrenamiento | por decidir en el Tratado 2 | Las imágenes miden ~2872×1504; a `imgsz=640` las lesiones periapicales quedan muy pequeñas. Recomiendo fijar **el mismo** `imgsz` (p. ej. 1024) para ambos modelos. |
| Licencia | verifícala en la ficha de Zenodo | Si no permite redistribución, en GitHub solo van `data.yaml`, `splits/*.txt`, CSV, evidencias y el enlace. |

---

## 3. Pipeline corregido (celdas de Colab)

### Celda 0 — Configuración

In [2]:
import os, io, json, shutil, zipfile, random, hashlib, re
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATASET_DIR = Path("/content/drive/MyDrive/UNIVERSIDAD/10MO SEMESTRE/ARTICULO II/DATASET")
ZIP_DRIVE_PATH    = DRIVE_DATASET_DIR / "training_data.zip"
OUT_ZIP_DRIVE     = DRIVE_DATASET_DIR / "DENTEX-YOLO-2clases.zip"
OUT_META_DRIVE    = DRIVE_DATASET_DIR / "metadata_dentex"

RAW_ROOT  = Path("/content/dentex_raw")
YOLO_ROOT = Path("/content/dentex_yolo")
SPLITS_DIR = YOLO_ROOT / "splits"

CLASS_NAMES = ["periapical_lesion", "impacted_tooth"]   # 0, 1
SEED = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

INCLUDE_NEGATIVES = True    # radiografías sin ninguna de las 2 clases (solo caries) como fondo
SIZE_MISMATCH_TOL = 0.01    # máx. 1 % de imágenes con tamaño real != tamaño del JSON

random.seed(SEED); np.random.seed(SEED)
OUT_META_DRIVE.mkdir(parents=True, exist_ok=True)
print("ZIP existe:", ZIP_DRIVE_PATH.exists())

Mounted at /content/drive
ZIP existe: True


### Celda 1 — Extraer SOLO la carpeta `quadrant-enumeration-disease`

In [3]:
SUBSET_TAG = "quadrant-enumeration-disease/"

with zipfile.ZipFile(ZIP_DRIVE_PATH) as zf:
    all_names = zf.namelist()
    members = [m for m in all_names if SUBSET_TAG in m and not m.endswith("/")]
    print("Entradas totales en el zip:", len(all_names), "| a extraer:", len(members))
    assert members, "No se encontró 'quadrant-enumeration-disease' en el zip; revisa all_names[:20]"

    if RAW_ROOT.exists():
        shutil.rmtree(RAW_ROOT)
    RAW_ROOT.mkdir(parents=True)
    for m in tqdm(members):
        zf.extract(m, RAW_ROOT)

print("Extraído en:", RAW_ROOT)

Entradas totales en el zip: 3615 | a extraer: 706


  0%|          | 0/706 [00:00<?, ?it/s]

Extraído en: /content/dentex_raw


### Celda 1b (opcional, evidencia para el artículo) — demostrar el problema de nombres repetidos

Lee solo las cabeceras de las imágenes directamente del zip (rápido, no extrae nada).

In [4]:
with zipfile.ZipFile(ZIP_DRIVE_PATH) as zf:
    sizes = defaultdict(dict)
    for n in zf.namelist():
        if n.lower().endswith(".png") and Path(n).name == "train_0.png":
            folder = Path(n).parts[-3]
            with zf.open(n) as fh:
                sizes[folder] = Image.open(fh).size
print("Tamaño de 'train_0.png' según la carpeta:")
for k, v in sizes.items():
    print(f"  {k:35s} {v}")
# Si los tamaños difieren entre carpetas => nombres iguales, imágenes distintas.

Tamaño de 'train_0.png' según la carpeta:
  quadrant                            (2955, 1316)
  unlabelled                          (2480, 1504)
  quadrant-enumeration-disease        (2850, 1316)
  quadrant_enumeration                (2955, 1316)


### Celda 2 — Localizar JSON e imágenes **de la misma carpeta**

In [5]:
ANN_JSON = next(RAW_ROOT.rglob("train_quadrant_enumeration_disease.json"))
IMG_DIR  = ANN_JSON.parent / "xrays"          # <- CLAVE: las imágenes viven junto al JSON
assert IMG_DIR.is_dir(), IMG_DIR

with open(ANN_JSON, "r", encoding="utf-8") as f:
    coco = json.load(f)

print("ANN_JSON:", ANN_JSON)
print("IMG_DIR :", IMG_DIR, "|", len(list(IMG_DIR.iterdir())), "archivos")
print("Imágenes en JSON:", len(coco["images"]), "| anotaciones:", len(coco["annotations"]))
print("Claves:", list(coco.keys()))

ANN_JSON: /content/dentex_raw/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json
IMG_DIR : /content/dentex_raw/training_data/quadrant-enumeration-disease/xrays | 705 archivos
Imágenes en JSON: 705 | anotaciones: 3529
Claves: ['images', 'annotations', 'categories_1', 'categories_2', 'categories_3']


### Celda 3 — Mapear categorías (nivel diagnóstico)

In [6]:
DIAG_CAT_KEY  = "categories_3"
ANN_CAT_FIELD = "category_id_3"
print(coco[DIAG_CAT_KEY])

def norm(s): return re.sub(r"[^a-z]+", " ", str(s).lower()).strip()

id_to_yolo = {}
for c in coco[DIAG_CAT_KEY]:
    n = norm(c["name"])
    if "periapical" in n:   id_to_yolo[c["id"]] = 0
    elif n == "impacted":   id_to_yolo[c["id"]] = 1     # exacto: evita falsas coincidencias
assert set(id_to_yolo.values()) == {0, 1}, id_to_yolo

name_by_id = {c["id"]: c["name"] for c in coco[DIAG_CAT_KEY]}
diag_counts = Counter(a[ANN_CAT_FIELD] for a in coco["annotations"])
for cid, n in sorted(diag_counts.items()):
    print(f"  {cid} {name_by_id[cid]:18s} {n:5d}{'  <- OBJETIVO' if cid in id_to_yolo else ''}")

[{'id': 0, 'name': 'Impacted', 'supercategory': 'Impacted'}, {'id': 1, 'name': 'Caries', 'supercategory': 'Caries'}, {'id': 2, 'name': 'Periapical Lesion', 'supercategory': 'Periapical Lesion'}, {'id': 3, 'name': 'Deep Caries', 'supercategory': 'Deep Caries'}]
  0 Impacted             604  <- OBJETIVO
  1 Caries              2189
  2 Periapical Lesion    158  <- OBJETIVO
  3 Deep Caries          578


Esperado (igual que antes): Impacted 604, Periapical Lesion 158.

### Celda 4 — Auditoría imagen ↔ anotación (versión corregida)

In [7]:
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
files_on_disk = {p.name for p in IMG_DIR.iterdir() if p.suffix.lower() in IMG_EXT}

imgs  = {im["id"]: im for im in coco["images"]}
names = [Path(im["file_name"]).name for im in imgs.values()]
assert len(names) == len(set(names)), "file_name duplicados dentro del JSON"

json_sin_archivo = [n for n in names if n not in files_on_disk]
archivo_sin_json = sorted(files_on_disk - set(names))
print("Imágenes en disco (misma carpeta):", len(files_on_disk))
print("JSON sin archivo:", len(json_sin_archivo), "| archivos sin JSON:", len(archivo_sin_json))
assert not json_sin_archivo, json_sin_archivo[:5]

records, n_mismatch = {}, 0
for iid, im in tqdm(imgs.items()):
    fname = Path(im["file_name"]).name
    p = IMG_DIR / fname
    with Image.open(p) as pil:                 # solo lee cabecera
        W, H = pil.size
    if (W, H) != (im["width"], im["height"]):
        n_mismatch += 1
    records[iid] = {"image_id": iid, "file_name": fname, "path": p,
                    "width": W, "height": H, "json_w": im["width"], "json_h": im["height"], "boxes": []}

print(f"Tamaño real != JSON: {n_mismatch}/{len(records)} ({100*n_mismatch/len(records):.1f} %)")
assert n_mismatch / len(records) < SIZE_MISMATCH_TOL, \
    "Demasiadas imágenes con tamaño distinto al JSON: el emparejamiento sigue mal."

for a in coco["annotations"]:
    cid = a[ANN_CAT_FIELD]
    if cid in id_to_yolo and a["image_id"] in records:
        records[a["image_id"]]["boxes"].append((id_to_yolo[cid], a["bbox"], a["id"]))
print("Imágenes emparejadas:", len(records))

Imágenes en disco (misma carpeta): 705
JSON sin archivo: 0 | archivos sin JSON: 0


  0%|          | 0/705 [00:00<?, ?it/s]

Tamaño real != JSON: 0/705 (0.0 %)
Imágenes emparejadas: 705


> En tu corrida anterior esta celda habría dado ~93 % de desajuste y habría **abortado**: ese es el aviso que faltaba.

### Celda 4b — Validar el formato del `bbox` contra el polígono

No se asume `[x, y, w, h]`: se comprueba con el polígono de `segmentation`.

In [8]:
def poly_bounds(seg):
    pts = np.concatenate([np.array(s, float).reshape(-1, 2) for s in seg if len(s) >= 6])
    return pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max()

TOL = 5.0  # px
res = Counter()
for a in coco["annotations"]:
    seg = a.get("segmentation")
    if not seg or not any(len(s) >= 6 for s in seg):
        res["sin_poligono"] += 1; continue
    ref = np.array(poly_bounds(seg))
    x, y, w, h = a["bbox"]
    as_xywh = np.array([x, y, x + w, y + h])
    as_xyxy = np.array([x, y, w, h])
    ok1 = np.abs(as_xywh - ref).max() <= TOL
    ok2 = np.abs(as_xyxy - ref).max() <= TOL
    res["xywh" if ok1 else "xyxy" if ok2 else "ninguno"] += 1
print(dict(res))

validos = res["xywh"] + res["xyxy"] + res["ninguno"]
BBOX_FORMAT = "xywh" if res["xywh"] >= res["xyxy"] else "xyxy"
print("BBOX_FORMAT =", BBOX_FORMAT, f"({max(res['xywh'], res['xyxy'])}/{validos} coinciden con el polígono)")
assert max(res["xywh"], res["xyxy"]) / validos > 0.95, \
    "El bbox no coincide con el polígono: derivar las cajas desde 'segmentation'."

def to_xyxy(b):
    x, y, p, q = b
    return (x, y, x + p, y + q) if BBOX_FORMAT == "xywh" else (x, y, p, q)

{'xywh': 3529}
BBOX_FORMAT = xywh (3529/3529 coinciden con el polígono)


### Celda 5 — Estadísticas

In [9]:
rows = []
for r in records.values():
    c = Counter(cl for cl, _, _ in r["boxes"])
    rows.append({"image_id": r["image_id"], "file_name": r["file_name"],
                 "width": r["width"], "height": r["height"],
                 "n_perio": c[0], "n_imp": c[1], "n_total": c[0] + c[1]})
df = pd.DataFrame(rows)
df["has_perio"], df["has_imp"], df["has_target"] = df.n_perio > 0, df.n_imp > 0, df.n_total > 0

resumen = pd.DataFrame({
    "clase": CLASS_NAMES,
    "imagenes_con_la_clase": [int(df.has_perio.sum()), int(df.has_imp.sum())],
    "instancias": [int(df.n_perio.sum()), int(df.n_imp.sum())],
})
print("Imágenes totales:", len(df), "| con >=1 clase objetivo:", int(df.has_target.sum()),
      "| ambas:", int((df.has_perio & df.has_imp).sum()), "| negativas:", int((~df.has_target).sum()))
display(resumen)
print("Resoluciones más frecuentes:", df.groupby(["width", "height"]).size().sort_values(ascending=False).head(5).to_dict())

df.to_csv(OUT_META_DRIVE / "estadisticas_por_imagen.csv", index=False)
resumen.to_csv(OUT_META_DRIVE / "resumen_clases_global.csv", index=False)

Imágenes totales: 705 | con >=1 clase objetivo: 326 | ambas: 44 | negativas: 379


,clase,imagenes_con_la_clase,instancias
0,periapical_lesion,116,158
1,impacted_tooth,254,604


Resoluciones más frecuentes: {(2872, 1504): 133, (2868, 1504): 38, (1976, 976): 25, (2884, 1504): 15, (3076, 1536): 8}


### Celda 6 — Conversión a YOLO

In [10]:
def boxes_to_yolo(boxes, W, H):
    lines, kept_xyxy, dropped = [], [], 0
    for cls, b, _ in boxes:
        x1, y1, x2, y2 = to_xyxy(b)
        x1, y1, x2, y2 = max(0.0, x1), max(0.0, y1), min(float(W), x2), min(float(H), y2)
        bw, bh = x2 - x1, y2 - y1
        if bw <= 1 or bh <= 1:
            dropped += 1; continue
        lines.append(f"{cls} {(x1+bw/2)/W:.6f} {(y1+bh/2)/H:.6f} {bw/W:.6f} {bh/H:.6f}")
        kept_xyxy.append((cls, x1, y1, x2, y2))
    return lines, kept_xyxy, dropped

total_dropped = 0
for r in records.values():
    r["yolo_lines"], r["ref_xyxy"], d = boxes_to_yolo(r["boxes"], r["width"], r["height"])
    total_dropped += d
print("Cajas degeneradas descartadas:", total_dropped)
ej = next(r for r in records.values() if r["yolo_lines"])
print("Ejemplo:", ej["file_name"], ej["yolo_lines"][:3])

Cajas degeneradas descartadas: 0
Ejemplo: train_673.png ['1 0.237609 0.633359 0.080175 0.205927', '1 0.743622 0.629179 0.064504 0.205167', '0 0.525823 0.684877 0.027451 0.268425']


### Celda 7 — Split único (seed 42)

In [11]:
from sklearn.model_selection import train_test_split

def stratum(r):
    c = {cl for cl, _, _ in r["boxes"]}
    return "ambas" if c == {0, 1} else "perio" if c == {0} else "imp" if c == {1} else "ninguna"

pool = [r for r in records.values() if (r["yolo_lines"] or INCLUDE_NEGATIVES)]
for r in pool: r["stratum"] = stratum(r)

cnt = Counter(r["stratum"] for r in pool)
print("Estratos:", dict(cnt))
ok   = [r for r in pool if cnt[r["stratum"]] >= 3]
left = [r for r in pool if cnt[r["stratum"]] < 3]
if left: print(f"AVISO: {len(left)} imágenes en estratos con <3 muestras -> train.")

pool_sorted = sorted(ok, key=lambda r: r["file_name"])
train, temp = train_test_split(pool_sorted, test_size=VAL_RATIO + TEST_RATIO, random_state=SEED,
                               stratify=[r["stratum"] for r in pool_sorted])
val, test = train_test_split(temp, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), random_state=SEED,
                             stratify=[r["stratum"] for r in temp])
train += left
for name, rs in [("train", train), ("val", val), ("test", test)]:
    for r in rs: r["split"] = name
print(f"train={len(train)} val={len(val)} test={len(test)} total={len(train)+len(val)+len(test)}")

rows = []
for name, rs in [("train", train), ("val", val), ("test", test)]:
    rows.append({"split": name, "imagenes": len(rs),
                 "imagenes_negativas": sum(1 for r in rs if not r["yolo_lines"]),
                 "inst_periapical": sum(1 for r in rs for cl, *_ in r["ref_xyxy"] if cl == 0),
                 "inst_impactado":  sum(1 for r in rs for cl, *_ in r["ref_xyxy"] if cl == 1)})
split_table = pd.DataFrame(rows); display(split_table)
display(pd.crosstab(pd.Series([r["stratum"] for r in train+val+test], name="estrato"),
                    pd.Series([r["split"] for r in train+val+test], name="split")))
split_table.to_csv(OUT_META_DRIVE / "tabla_split_clases.csv", index=False)

Estratos: {'ambas': 44, 'imp': 210, 'ninguna': 379, 'perio': 72}
train=493 val=106 test=106 total=705


,split,imagenes,imagenes_negativas,inst_periapical,inst_impactado
0,train,493,265,107,428
1,val,106,57,24,83
2,test,106,57,27,93


split,test,train,val
estrato,,,
ambas,7,31,6
imp,31,147,32
ninguna,57,265,57
perio,11,50,11


### Celda 8 — Escribir el dataset YOLO + evidencia del split

In [12]:
if YOLO_ROOT.exists(): shutil.rmtree(YOLO_ROOT)
for s in ["train", "val", "test"]:
    (YOLO_ROOT / "images" / s).mkdir(parents=True)
    (YOLO_ROOT / "labels" / s).mkdir(parents=True)
SPLITS_DIR.mkdir(parents=True)

for s, rs in [("train", train), ("val", val), ("test", test)]:
    for r in tqdm(rs, desc=s):
        shutil.copy2(r["path"], YOLO_ROOT / "images" / s / r["file_name"])
        txt = "\n".join(r["yolo_lines"]) + ("\n" if r["yolo_lines"] else "")   # vacío = negativa
        (YOLO_ROOT / "labels" / s / (Path(r["file_name"]).stem + ".txt")).write_text(txt)

hashes = {}
for s, rs in [("train", train), ("val", val), ("test", test)]:
    content = "\n".join(sorted(r["file_name"] for r in rs)) + "\n"
    (SPLITS_DIR / f"{s}.txt").write_text(content)
    hashes[s] = hashlib.sha256(content.encode()).hexdigest()

(YOLO_ROOT / "data.yaml").write_text(
    f"path: {YOLO_ROOT}\ntrain: images/train\nval: images/val\ntest: images/test\n"
    f"nc: {len(CLASS_NAMES)}\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASS_NAMES))
)

meta = {"source": "DENTEX (Zenodo 7812323), carpeta quadrant-enumeration-disease",
        "json": ANN_JSON.name, "bbox_format_detected": BBOX_FORMAT, "classes": CLASS_NAMES,
        "seed": SEED, "ratios": [TRAIN_RATIO, VAL_RATIO, TEST_RATIO],
        "include_negatives": INCLUDE_NEGATIVES, "stratification": "combinacion de clases por imagen",
        "n_images": {"train": len(train), "val": len(val), "test": len(test)},
        "sha256_split_lists": hashes, "same_split_for_models": True}
(YOLO_ROOT / "split_metadata.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

train:   0%|          | 0/493 [00:00<?, ?it/s]

val:   0%|          | 0/106 [00:00<?, ?it/s]

test:   0%|          | 0/106 [00:00<?, ?it/s]

{
  "source": "DENTEX (Zenodo 7812323), carpeta quadrant-enumeration-disease",
  "json": "train_quadrant_enumeration_disease.json",
  "bbox_format_detected": "xywh",
  "classes": [
    "periapical_lesion",
    "impacted_tooth"
  ],
  "seed": 42,
  "ratios": [
    0.7,
    0.15,
    0.15
  ],
  "include_negatives": true,
  "stratification": "combinacion de clases por imagen",
  "n_images": {
    "train": 493,
    "val": 106,
    "test": 106
  },
  "sha256_split_lists": {
    "train": "5c2f945aea48c55fa4db3a14d1e6e5053598ba5ded4587c0200c02334dbe3d36",
    "val": "d79395131d954e0879e73ba3dfb06f4b33ae582febe20154591d579c31f9446c",
    "test": "9a8f58210b497e25b9198e6070890c2a9292f2e86ea3635645eb54a585f499a1"
  },
  "same_split_for_models": true
}


### Celda 9 — Verificación numérica *round-trip* (obligatoria)

Reconstruye las cajas desde los `.txt` escritos y las compara con el JSON original.

In [13]:
max_err, n_boxes, n_bad = 0.0, 0, 0
for r in train + val + test:
    lab = YOLO_ROOT / "labels" / r["split"] / (Path(r["file_name"]).stem + ".txt")
    lines = [l.split() for l in lab.read_text().splitlines() if l.strip()]
    assert len(lines) == len(r["ref_xyxy"]), r["file_name"]
    W, H = r["width"], r["height"]
    for (c, xc, yc, w, h), (c0, x1, y1, x2, y2) in zip(lines, r["ref_xyxy"]):
        xc, yc, w, h = float(xc)*W, float(yc)*H, float(w)*W, float(h)*H
        rec = np.array([xc - w/2, yc - h/2, xc + w/2, yc + h/2])
        err = np.abs(rec - np.array([x1, y1, x2, y2])).max()
        max_err = max(max_err, err); n_boxes += 1
        n_bad += int(int(c) != c0 or err > 1.0)
print(f"Cajas verificadas: {n_boxes} | error máximo: {max_err:.4f} px | inconsistentes: {n_bad}")
assert n_bad == 0

Cajas verificadas: 762 | error máximo: 0.0021 px | inconsistentes: 0


### Celda 10 — Verificación visual lado a lado (JSON crudo vs. label YOLO)

In [14]:
COLORS = {0: (255, 60, 60), 1: (60, 220, 60)}
ann_by_img = defaultdict(list)
for a in coco["annotations"]:
    if a[ANN_CAT_FIELD] in id_to_yolo:
        ann_by_img[a["image_id"]].append(a)

def draw_json(r):
    im = cv2.cvtColor(cv2.imread(str(r["path"])), cv2.COLOR_BGR2RGB)
    for a in ann_by_img[r["image_id"]]:
        c = id_to_yolo[a[ANN_CAT_FIELD]]
        x1, y1, x2, y2 = map(int, to_xyxy(a["bbox"]))
        cv2.rectangle(im, (x1, y1), (x2, y2), COLORS[c], 6)
        cv2.putText(im, CLASS_NAMES[c], (x1, max(30, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1.6, COLORS[c], 4)
    return im

def draw_yolo(r):
    im = cv2.cvtColor(cv2.imread(str(YOLO_ROOT / "images" / r["split"] / r["file_name"])), cv2.COLOR_BGR2RGB)
    H, W = im.shape[:2]
    lab = YOLO_ROOT / "labels" / r["split"] / (Path(r["file_name"]).stem + ".txt")
    for l in lab.read_text().splitlines():
        c, xc, yc, w, h = l.split(); c = int(c)
        xc, yc, w, h = float(xc)*W, float(yc)*H, float(w)*W, float(h)*H
        x1, y1, x2, y2 = int(xc - w/2), int(yc - h/2), int(xc + w/2), int(yc + h/2)
        cv2.rectangle(im, (x1, y1), (x2, y2), COLORS[c], 6)
        cv2.putText(im, CLASS_NAMES[c], (x1, max(30, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1.6, COLORS[c], 4)
    return im

random.seed(SEED)
allr = train + val + test
sel = []
for st in ["perio", "imp", "ambas"]:
    sel += random.sample([r for r in allr if r["stratum"] == st], 2)

fig, axs = plt.subplots(len(sel), 2, figsize=(18, 5.2 * len(sel)))
for row, r in zip(axs, sel):
    row[0].imshow(draw_json(r)); row[0].set_title(f"{r['file_name']} — JSON original", fontsize=9)
    row[1].imshow(draw_yolo(r)); row[1].set_title(f"{r['file_name']} — label YOLO ({r['split']})", fontsize=9)
    for ax in row: ax.axis("off")
plt.tight_layout()
fig.savefig(OUT_META_DRIVE / "verificacion_visual_cajas.png", dpi=110)
plt.show()

Output hidden; open in https://colab.research.google.com to view.

**Criterio de aprobación:** los impactados (verde) deben rodear terceros molares incluidos en hueso; las lesiones periapicales (rojo) deben ubicarse sobre el ápice de una raíz. Ambos paneles deben ser idénticos.

### Celda 11 — No-fuga y split idéntico para ambos modelos

In [15]:
sets = {n: set((SPLITS_DIR / f"{n}.txt").read_text().split()) for n in ["train", "val", "test"]}
assert not (sets["train"] & sets["val"]) and not (sets["train"] & sets["test"]) and not (sets["val"] & sets["test"]), "FUGA"
for n in sets:
    on_disk = {p.name for p in (YOLO_ROOT / "images" / n).iterdir()}
    assert on_disk == sets[n], f"Lista y carpeta difieren en {n}"
    assert len(list((YOLO_ROOT / "labels" / n).glob("*.txt"))) == len(sets[n])
print("OK: splits disjuntos y coherentes con carpetas/labels.")

yaml_hash = hashlib.sha256((YOLO_ROOT / "data.yaml").read_bytes()).hexdigest()
evidence = {"data_yaml_sha256": yaml_hash, "split_list_sha256": hashes,
            "models": {"yolo11n": {"data": "data.yaml", "seed": SEED},
                       "yolo26n": {"data": "data.yaml", "seed": SEED}}}
(YOLO_ROOT / "split_evidence.json").write_text(json.dumps(evidence, indent=2))
print(json.dumps(evidence, indent=2))

OK: splits disjuntos y coherentes con carpetas/labels.
{
  "data_yaml_sha256": "1621825915e761895e379b67ec63ed25f6c75e4e608b5d29a64db75d73a625b1",
  "split_list_sha256": {
    "train": "5c2f945aea48c55fa4db3a14d1e6e5053598ba5ded4587c0200c02334dbe3d36",
    "val": "d79395131d954e0879e73ba3dfb06f4b33ae582febe20154591d579c31f9446c",
    "test": "9a8f58210b497e25b9198e6070890c2a9292f2e86ea3635645eb54a585f499a1"
  },
  "models": {
    "yolo11n": {
      "data": "data.yaml",
      "seed": 42
    },
    "yolo26n": {
      "data": "data.yaml",
      "seed": 42
    }
  }
}


### Celda 12 — Empaquetado y subida a Drive (carpeta DATASET)

In [16]:
tmp_zip = Path("/content/DENTEX-YOLO-2clases.zip")
with zipfile.ZipFile(tmp_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in YOLO_ROOT.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(YOLO_ROOT))
shutil.copy2(tmp_zip, OUT_ZIP_DRIVE)

for fn in ["data.yaml", "split_metadata.json", "split_evidence.json"]:
    shutil.copy2(YOLO_ROOT / fn, OUT_META_DRIVE / fn)
shutil.copytree(SPLITS_DIR, OUT_META_DRIVE / "splits", dirs_exist_ok=True)

print("Dataset YOLO:", OUT_ZIP_DRIVE, f"({OUT_ZIP_DRIVE.stat().st_size/1e9:.2f} GB)")
print("Metadatos   :", OUT_META_DRIVE)
print("train/val/test:", len(train), len(val), len(test))

Dataset YOLO: /content/drive/MyDrive/UNIVERSIDAD/10MO SEMESTRE/ARTICULO II/DATASET/DENTEX-YOLO-2clases.zip (2.15 GB)
Metadatos   : /content/drive/MyDrive/UNIVERSIDAD/10MO SEMESTRE/ARTICULO II/DATASET/metadata_dentex
train/val/test: 493 106 106


---

## 4. Qué debes esperar al correrlo

- Celda 4: desajuste de tamaño ≈ 0 % (antes 93 %).
- Celda 4b: casi todos los `bbox` coinciden con el polígono como `xywh`.
- Celda 5: mismos conteos de instancias (158 periapicales, 604 impactados); cambia el número de imágenes si `INCLUDE_NEGATIVES=True` (705 en total vs. 326).
- Celda 9: error máximo < 0.01 px.
- Celda 10: cajas sobre anatomía correcta.

**Importante:** los hashes y los conteos por split cambiarán respecto a la corrida anterior. Descarta `DENTEX-YOLO-2clases.zip` y `metadata_dentex/` antiguos (se sobrescriben con los mismos nombres) y no entrenes con nada generado antes de esta corrección.

## 5. Si algo aún falla

| Síntoma | Causa probable | Acción |
|---|---|---|
| Celda 4 aborta por tamaño | La carpeta extraída no es la de *disease* | Revisar `IMG_DIR` y el árbol de `RAW_ROOT` |
| Celda 4b no pasa el 95 % | `bbox` no fiable | Derivar la caja de `poly_bounds(segmentation)` |
| Cajas bien en JSON pero mal en YOLO | Error de escritura/lectura | La celda 9 lo detecta numéricamente |
| Cajas correctas pero "raras" anatómicamente | Anotación ruidosa | Ver las imágenes del paso 10 y reportarlo como limitación |

## 6. Notas para el repositorio

- Subir: `data.yaml`, `splits/*.txt`, `split_metadata.json`, `split_evidence.json`, CSV, este notebook y el enlace a https://zenodo.org/records/7812323. No subir radiografías si la licencia no lo permite.
- Citar el dataset DENTEX según la ficha de Zenodo.
- La nota sobre nombres repetidos entre carpetas conviene dejarla en el README (ayuda a la reproducibilidad).